# LeetCode 3500 — Minimum Cost to Divide Array Into Subarrays bằng Python

- Link: https://leetcode.com/problems/minimum-cost-to-divide-array-into-subarrays/
- Độ khó: **Hard**
- Chủ đề: **DP chia đoạn**, mẹo **khử chỉ số thứ tự đoạn** bằng tổng hậu tố, **tổng tiền tố**, **Convex Hull Trick** (bao lồi các đường thẳng) / Li Chao

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minimumCost(self, nums: List[int], cost: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Viết chi phí đoạn thứ `i` = `(P[r] + k·i) · (C[r] − C[l−1])` với `P`, `C` là tổng tiền tố của `nums`, `cost`.
2. Nhận ra chỉ số `i` (thứ tự đoạn) làm DP cần thêm một chiều ⇒ `O(n^3)`.
3. **Mẹo then chốt:** `Σ_i k·i·(cost đoạn i) = k · Σ_{mỗi đoạn} (tổng cost từ đầu đoạn tới cuối mảng)` ⇒ xóa chiều `i`, còn DP `O(n^2)`.
4. Biến DP thành bài **min của các đường thẳng** tại điểm `x = P[r]` ⇒ Convex Hull Trick `O(n)` (độ dốc và điểm truy vấn đều đơn điệu).
5. Kiểm chứng bằng vét cạn mọi cách chia.

## 2. Đề bài & ví dụ

Cho `nums`, `cost` cùng độ dài `n` và số `k`. Chia `nums` thành các đoạn con liên tiếp. Đoạn thứ `i` (đánh số từ 1) là `nums[l..r]` có chi phí:

`(nums[0] + … + nums[r] + k·i) · (cost[l] + … + cost[r])`.

Trả về tổng chi phí nhỏ nhất.

**Ví dụ 1:** `nums = [3,1,4], cost = [4,6,6], k = 1` → `110` (chia `[3,1] | [4]`: `(4+1)·10 + (8+2)·6 = 50 + 60`).

**Ví dụ 2:** `nums = [4,8,5,1,14,2,2,12,1], cost = [7,2,8,4,2,2,1,1,2], k = 7` → `985`.

**Ràng buộc:** `1 ≤ n ≤ 1000`, `1 ≤ nums[i], cost[i], k ≤ 1000`.

In [1]:
from typing import List
examples = [([3, 1, 4], [4, 6, 6], 1, 110), ([4, 8, 5, 1, 14, 2, 2, 12, 1], [7, 2, 8, 4, 2, 2, 1, 1, 2], 7, 985)]
for nums, cost, k, exp in examples:
    print(nums, cost, k, '-> kỳ vọng', exp)

[3, 1, 4] [4, 6, 6] 1 -> kỳ vọng 110
[4, 8, 5, 1, 14, 2, 2, 12, 1] [7, 2, 8, 4, 2, 2, 1, 1, 2] 7 -> kỳ vọng 985


## 3. Kiến thức nền

### 3.1. Tổng tiền tố

`P[j] = nums[0] + … + nums[j−1]`, `C[j]` tương tự; tổng đoạn `[l, r]` của `cost` là `C[r+1] − C[l]`.

In [2]:
def prefix(a):
    p = [0]
    for v in a:
        p.append(p[-1] + v)
    return p

print(prefix([3, 1, 4]), prefix([4, 6, 6]))

[0, 3, 4, 8] [0, 4, 10, 16]


### 3.2. Đổi thứ tự cộng: `Σ i·w_i`

Với các đoạn có tổng cost `w_1, w_2, …, w_m`: `Σ i·w_i = Σ_{j=1}^{m} (w_j + w_{j+1} + … + w_m)` (mỗi `w_i` được đếm `i` lần — một lần cho mỗi `j ≤ i`). Mà `w_j + … + w_m` = **tổng cost từ đầu đoạn `j` tới hết mảng** = `C[n] − C[l_j]`.

In [3]:
w = [10, 6]
print(sum((i + 1) * x for i, x in enumerate(w)), sum(sum(w[j:]) for j in range(len(w))))

22 22


### 3.3. DP chia đoạn

`dp[r]` = chi phí nhỏ nhất cho tiền tố `nums[0..r−1]`; đoạn cuối `[l, r−1]`: `dp[r] = min_l dp[l] + chi phí(đoạn)`.

### 3.4. Min của các đường thẳng

Nếu `dp[r] = min_l (a_l · x_r + b_l) + c_r` thì mỗi `l` là một **đường thẳng** `y = a_l x + b_l`; cần giá trị nhỏ nhất tại `x = x_r`. Khi độ dốc giảm dần và `x` tăng dần, dùng **deque bao lồi** (Convex Hull Trick) `O(1)` khấu hao mỗi thao tác.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tách chi phí đoạn `i`:

`(P[r] + k·i)·W_i = P[r]·W_i + k·i·W_i`.

Phần đầu chỉ phụ thuộc đoạn. Phần sau phụ thuộc **thứ tự** `i` — nhưng theo 3.2, cộng trên mọi đoạn: `k·Σ i·W_i = k·Σ_j (C[n] − C[l_j])`. Vậy tổng chi phí

`= Σ_{đoạn [l, r]} ( P[r+1]·(C[r+1] − C[l]) + k·(C[n] − C[l]) )`

— **không còn** chỉ số `i`!

Tính tay ví dụ 1, chia `[3,1] | [4]`: `P = [0,3,4,8]`, `C = [0,4,10,16]`.

- Đoạn `[0,1]`: `4·10 + 1·(16 − 0) = 56`.
- Đoạn `[2,2]`: `8·6 + 1·(16 − 10) = 54`.
- Tổng `110` ✓ (khớp `50 + 60`, chỉ phân bổ khác).

In [4]:
P, C = prefix([3, 1, 4]), prefix([4, 6, 6])
k = 1
seg = lambda l, r: P[r + 1] * (C[r + 1] - C[l]) + k * (C[3] - C[l])
print(seg(0, 1), seg(2, 2), seg(0, 1) + seg(2, 2))

56 54 110


## 5. Cách 1 — Brute Force: thử mọi cách chia

In [5]:
def min_cost_brute(nums, cost, k):
    n = len(nums)
    P, C = prefix(nums), prefix(cost)
    best = None
    for mask in range(1 << (n - 1)):
        cuts = [0] + [i + 1 for i in range(n - 1) if mask >> i & 1] + [n]
        total = 0
        for idx in range(len(cuts) - 1):
            l, r = cuts[idx], cuts[idx + 1] - 1
            total += (P[r + 1] + k * (idx + 1)) * (C[r + 1] - C[l])
        best = total if best is None else min(best, total)
    return best

for nums, cost, k, exp in examples:
    print(min_cost_brute(nums, cost, k), '| kỳ vọng', exp)

110 | kỳ vọng 110
985 | kỳ vọng 985


**Độ phức tạp:** `O(2^{n−1} · n)` — chỉ với `n ≤ 18`.

**Nút thắt:** chi phí phụ thuộc thứ tự đoạn `i` ⇒ DP "tự nhiên" phải nhớ thêm `i`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 (khử `i`):** tổng chi phí = `Σ_{đoạn [l, r]} P[r+1]·(C[r+1] − C[l]) + k·(C[n] − C[l])` (mục 4).

**Bước 2 (DP `O(n^2)`):** `dp[0] = 0`, `dp[r] = min_{0 ≤ l < r} dp[l] + P[r]·(C[r] − C[l]) + k·(C[n] − C[l])` (chỉ số `r` ở đây là số phần tử đã chia).

**Bước 3 (CHT):** khai triển

`dp[r] = P[r]·C[r] + k·C[n] + min_l ( −C[l]·P[r] + dp[l] − k·C[l] )`.

Đường thẳng thứ `l`: độ dốc `−C[l]` (giảm dần vì `C` tăng), tung độ gốc `dp[l] − k·C[l]`; truy vấn tại `x = P[r]` (tăng dần) ⇒ deque bao lồi, `O(n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP có chiều "số đoạn" (`O(n^3)`)

`f[j][r]` = chi phí nhỏ nhất chia `nums[0..r−1]` thành đúng `j` đoạn. Đúng theo định nghĩa gốc, nhưng chậm.

In [6]:
def min_cost_cubic(nums, cost, k):
    n = len(nums)
    P, C = prefix(nums), prefix(cost)
    INF = float('inf')
    f = [[INF] * (n + 1) for _ in range(n + 1)]
    f[0][0] = 0
    for j in range(1, n + 1):
        for r in range(j, n + 1):
            best = INF
            for l in range(j - 1, r):
                if f[j - 1][l] < INF:
                    v = f[j - 1][l] + (P[r] + k * j) * (C[r] - C[l])
                    if v < best:
                        best = v
            f[j][r] = best
    return min(f[j][n] for j in range(1, n + 1))

for nums, cost, k, exp in examples:
    print(min_cost_cubic(nums, cost, k), '| kỳ vọng', exp)

110 | kỳ vọng 110
985 | kỳ vọng 985


**Độ phức tạp:** `O(n^3)` ≈ `1.7·10^8` với `n = 1000` — quá chậm cho Python.

### 7.2. Cách 3 — DP `O(n^2)` sau khi khử thứ tự đoạn (lời giải chính)

In [7]:
def min_cost_quadratic(nums, cost, k):
    n = len(nums)
    P, C = prefix(nums), prefix(cost)
    INF = float('inf')
    dp = [INF] * (n + 1)
    dp[0] = 0
    Cn = C[n]
    for r in range(1, n + 1):
        pr, cr = P[r], C[r]
        best = INF
        for l in range(r):
            v = dp[l] + pr * (cr - C[l]) + k * (Cn - C[l])
            if v < best:
                best = v
        dp[r] = best
    return dp[n]

for nums, cost, k, exp in examples:
    print(min_cost_quadratic(nums, cost, k), '| kỳ vọng', exp)

110 | kỳ vọng 110
985 | kỳ vọng 985


**Độ phức tạp:** `O(n^2)` = `5·10^5` bước với `n = 1000`.

### 7.3. Cách 4 — Convex Hull Trick đơn điệu (`O(n)`)

Đường `l`: `y = m_l x + b_l` với `m_l = −C[l]`, `b_l = dp[l] − k·C[l]`. Thêm đường theo `l` tăng (độ dốc giảm), truy vấn `x = P[r]` tăng ⇒ deque: bỏ đường đầu khi đường thứ hai tốt hơn tại `x`, bỏ đường cuối khi nó trở nên thừa.

In [8]:
def min_cost_cht(nums, cost, k):
    n = len(nums)
    P, C = prefix(nums), prefix(cost)
    from collections import deque
    hull = deque()                      # các cặp (m, b)

    def bad(l1, l2, l3):
        # l2 thừa nếu giao điểm (l1, l3) nằm "trước" giao điểm (l1, l2)
        (m1, b1), (m2, b2), (m3, b3) = l1, l2, l3
        return (b3 - b1) * (m1 - m2) <= (b2 - b1) * (m1 - m3)

    def add(m, b):
        line = (m, b)
        while len(hull) >= 2 and bad(hull[-2], hull[-1], line):
            hull.pop()
        hull.append(line)

    def query(x):
        while len(hull) >= 2 and hull[1][0] * x + hull[1][1] <= hull[0][0] * x + hull[0][1]:
            hull.popleft()
        m, b = hull[0]
        return m * x + b

    dp = [0] * (n + 1)
    add(-C[0], dp[0] - k * C[0])
    for r in range(1, n + 1):
        dp[r] = P[r] * C[r] + k * C[n] + query(P[r])
        add(-C[r], dp[r] - k * C[r])
    return dp[n]

for nums, cost, k, exp in examples:
    print(min_cost_cht(nums, cost, k), '| kỳ vọng', exp)

110 | kỳ vọng 110
985 | kỳ vọng 985


Điều kiện `bad` dùng phép nhân chéo (số nguyên, không chia) để tránh sai số. Vì `cost[i] ≥ 1` nên các độ dốc **giảm ngặt**, không có hai đường song song.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1: in `dp[r]`, đoạn cuối tối ưu và từng thành phần chi phí.

In [9]:
def trace(nums, cost, k):
    n = len(nums)
    P, C = prefix(nums), prefix(cost)
    print('P =', P, '| C =', C)
    INF = float('inf')
    dp, arg = [0] + [INF] * n, [None] * (n + 1)
    for r in range(1, n + 1):
        for l in range(r):
            part1 = P[r] * (C[r] - C[l])
            part2 = k * (C[n] - C[l])
            v = dp[l] + part1 + part2
            print(f"  dp[{r}] thử đoạn cuối [{l}, {r-1}]: dp[{l}]={dp[l]} + {part1} + {part2} = {v}")
            if v < dp[r]:
                dp[r], arg[r] = v, l
        print(f"=> dp[{r}] = {dp[r]} (đoạn cuối bắt đầu tại {arg[r]})")
    segs, r = [], n
    while r:
        segs.append(nums[arg[r]:r]); r = arg[r]
    print('cách chia:', segs[::-1], '| tổng =', dp[n])

trace(*examples[0][:3])

P = [0, 3, 4, 8] | C = [0, 4, 10, 16]
  dp[1] thử đoạn cuối [0, 0]: dp[0]=0 + 12 + 16 = 28
=> dp[1] = 28 (đoạn cuối bắt đầu tại 0)
  dp[2] thử đoạn cuối [0, 1]: dp[0]=0 + 40 + 16 = 56
  dp[2] thử đoạn cuối [1, 1]: dp[1]=28 + 24 + 12 = 64
=> dp[2] = 56 (đoạn cuối bắt đầu tại 0)
  dp[3] thử đoạn cuối [0, 2]: dp[0]=0 + 128 + 16 = 144
  dp[3] thử đoạn cuối [1, 2]: dp[1]=28 + 96 + 12 = 136
  dp[3] thử đoạn cuối [2, 2]: dp[2]=56 + 48 + 6 = 110
=> dp[3] = 110 (đoạn cuối bắt đầu tại 2)
cách chia: [[3, 1], [4]] | tổng = 110


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def minimumCost(self, nums: List[int], cost: List[int], k: int) -> int:
        n = len(nums)
        P = [0] * (n + 1)
        C = [0] * (n + 1)
        for i in range(n):
            P[i + 1] = P[i] + nums[i]
            C[i + 1] = C[i] + cost[i]
        Cn = C[n]
        dp = [0] * (n + 1)
        for r in range(1, n + 1):
            pr, cr = P[r], C[r]
            best = None
            for l in range(r):
                v = dp[l] + pr * (cr - C[l]) + k * (Cn - C[l])
                if best is None or v < best:
                    best = v
            dp[r] = best
        return dp[n]

for nums, cost, k, exp in examples:
    got = Solution().minimumCost(nums, cost, k)
    print(got, 'OK' if got == exp else 'SAI')

110 OK
985 OK


## 10. Vì sao thuật toán đúng

1. **Đẳng thức khử thứ tự:** với các đoạn có tổng cost `W_1..W_m`, `Σ_i i·W_i = Σ_j Σ_{i ≥ j} W_i`, và `Σ_{i ≥ j} W_i = C[n] − C[l_j]`. Do đó tổng chi phí của **mọi** cách chia bằng `Σ_j (P[r_j+1]·W_j + k·(C[n] − C[l_j]))` — chỉ phụ thuộc từng đoạn riêng lẻ.
2. **DP chia đoạn** trên chi phí "cục bộ" này đúng vì cách chia tối ưu của tiền tố kết hợp đoạn cuối cho cách chia tối ưu (cấu trúc con tối ưu, chi phí cộng tính).
3. **CHT:** `dp[r] − (P[r]C[r] + kC[n]) = min_l (m_l·P[r] + b_l)`; deque giữ đúng bao dưới của các đường đã thêm; vì `x` tăng dần, đường tối ưu chỉ di chuyển về phía sau ⇒ bỏ đầu deque an toàn.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^n n)` | `O(n)` |
| 2. DP có số đoạn | `O(n^3)` | `O(n^2)` |
| 3. DP khử thứ tự | `O(n^2)` | `O(n)` |
| 4. CHT | `O(n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'cubic': min_cost_cubic, 'quadratic': min_cost_quadratic, 'cht': min_cost_cht,
         'submit': lambda a, c, k: Solution().minimumCost(a, c, k)}
edge = [([5], [3], 2, (5 + 2) * 3), ([1, 1], [1, 1], 1000, None), ([1000] * 5, [1000] * 5, 1000, None)]
for nums, cost, k, exp in examples + edge:
    b = min_cost_brute(nums, cost, k)
    if exp is not None:
        assert b == exp
    for name, f in impls.items():
        assert f(nums, cost, k) == b, (name, nums, cost, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [12]:
import random
random.seed(3500)
cnt = 0
for _ in range(400):
    n = random.randint(1, 10)
    nums = [random.randint(1, 20) for _ in range(n)]
    cost = [random.randint(1, 20) for _ in range(n)]
    k = random.randint(1, 30)
    exp = min_cost_brute(nums, cost, k)
    for name, f in impls.items():
        assert f(nums, cost, k) == exp, (name, nums, cost, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test ngẫu nhiên.')

Khớp vét cạn trên 400 bộ test ngẫu nhiên.


### 12.3. Ca lớn (`n = 1000`)

In [13]:
import time
random.seed(10)
for k in (1, 1000):
    nums = [random.randint(1, 1000) for _ in range(1000)]
    cost = [random.randint(1, 1000) for _ in range(1000)]
    t = time.time(); a = Solution().minimumCost(nums, cost, k); t1 = time.time() - t
    t = time.time(); b = min_cost_cht(nums, cost, k); t2 = time.time() - t
    assert a == b
    print(f"k={k}: {a} | O(n^2) {t1:.2f}s | CHT {t2:.4f}s")

k=1: 127294626050 | O(n^2) 0.07s | CHT 0.0009s


k=1000: 138195790114 | O(n^2) 0.07s | CHT 0.0012s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng tổng `nums` **của đoạn** thay vì **tiền tố** đến cuối đoạn

In [14]:
def wrong_segment_sum(nums, cost, k):
    n = len(nums)
    P, C = prefix(nums), prefix(cost)
    best = None
    for mask in range(1 << (n - 1)):
        cuts = [0] + [i + 1 for i in range(n - 1) if mask >> i & 1] + [n]
        tot = sum((P[cuts[j + 1]] - P[cuts[j]] + k * (j + 1)) * (C[cuts[j + 1]] - C[cuts[j]])   # SAI
                  for j in range(len(cuts) - 1))
        best = tot if best is None else min(best, tot)
    return best

nums, cost, k, exp = examples[0]
print('sai :', wrong_segment_sum(nums, cost, k), '| đúng:', exp)

sai : 76 | đúng: 110


**Vì sao sai:** đề dùng `nums[0] + … + nums[r]` — tổng **từ đầu mảng**.

### Lỗi 2 — DP `O(n^2)` nhưng quên phần `k·i` (hoặc cộng `k` thay vì `k·(C[n] − C[l])`)

In [15]:
def wrong_drop_k(nums, cost, k):
    n = len(nums)
    P, C = prefix(nums), prefix(cost)
    dp = [0] + [float('inf')] * n
    for r in range(1, n + 1):
        for l in range(r):
            dp[r] = min(dp[r], dp[l] + (P[r] + k) * (C[r] - C[l]))       # SAI: luôn coi i = 1
    return dp[n]

nums, cost, k, exp = examples[1]
print('sai :', wrong_drop_k(nums, cost, k), '| đúng:', exp)

sai : 777 | đúng: 985


**Vì sao sai:** đoạn thứ `i` nhân `k·i`, không phải `k`; phải dùng mẹo tổng hậu tố để biểu diễn đúng.

### Lỗi 3 — Tham lam chia thành từng phần tử riêng (hoặc không chia gì)

In [16]:
def all_singletons(nums, cost, k):
    P = prefix(nums)
    return sum((P[i + 1] + k * (i + 1)) * cost[i] for i in range(len(nums)))

def no_split(nums, cost, k):
    return (sum(nums) + k) * sum(cost)

nums, cost, k, exp = examples[1]
print('từng phần tử:', all_singletons(nums, cost, k), '| không chia:', no_split(nums, cost, k), '| tối ưu:', exp)

từng phần tử: 1316 | không chia: 1624 | tối ưu: 985


**Vì sao sai:** chia nhiều đoạn làm `nums` tiền tố nhỏ hơn cho các đoạn đầu nhưng phạt `k·i` tăng; cần cân bằng bằng DP.

### Lỗi 4 — CHT dùng phép chia số thực để so giao điểm

In [17]:
def intersect_x(l1, l2):
    (m1, b1), (m2, b2) = l1, l2
    return (b2 - b1) / (m1 - m2)

l1, l2, l3 = (-1, 10**17), (-2, 2 * 10**17 - 1), (-3, 3 * 10**17 - 1)
# l2 thừa <=> giao(l1, l3) <= giao(l1, l2); thật ra 1e17 - 0.5 <= 1e17 - 1 là SAI
print('so bằng số thực :', intersect_x(l1, l3) <= intersect_x(l1, l2))
print('so bằng nhân chéo:', (l3[1] - l1[1]) * (l1[0] - l2[0]) <= (l2[1] - l1[1]) * (l1[0] - l3[0]))

so bằng số thực : True
so bằng nhân chéo: False


**Vì sao sai:** giá trị cỡ `10^{12}`–`10^{17}` làm phép chia số thực mất chính xác; nhân chéo số nguyên luôn đúng (Python không tràn).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Chi phí đoạn phụ thuộc **thứ tự đoạn** dạng `i · w_i` ⇒ đổi thành tổng hậu tố `Σ_j (tổng từ đoạn j tới cuối)` để khử chiều `i`.
- DP chia đoạn với chi phí `a(l)·b(r) + …` ⇒ **Convex Hull Trick** / **Li Chao**.
- Độ dốc và điểm truy vấn đơn điệu ⇒ deque `O(n)`; không đơn điệu ⇒ Li Chao `O(n log)`.
- `n ≤ 1000` ⇒ `O(n^2)` thường đủ — CHT là phần mở rộng.

## 15. Biến thể của bài

1. Giới hạn **số đoạn tối đa** `m` ⇒ lại cần chiều số đoạn (`O(n^2 m)` hoặc CHT + chia để trị).
2. Phạt `k·i^2` ⇒ mẹo hậu tố không còn đơn giản (`i^2 = Σ (2j − 1)`) ⇒ có thể viết `Σ_j (2j−1)·(tổng từ j)`, vẫn phụ thuộc `j`.
3. Truy vấn `x` không đơn điệu ⇒ Li Chao tree.
4. Bài liên quan: LC 1478, LC 2478, "Frog 3" (CHT kinh điển).

In [18]:
# Biến thể 3: Li Chao tree (không cần đơn điệu), áp dụng cho bài gốc để kiểm tra
class LiChao:
    def __init__(self, lo, hi):
        self.lo, self.hi = lo, hi
        self.tree = {}

    def _f(self, line, x):
        return line[0] * x + line[1]

    def add(self, line, node=1, lo=None, hi=None):
        lo = self.lo if lo is None else lo
        hi = self.hi if hi is None else hi
        cur = self.tree.get(node)
        if cur is None:
            self.tree[node] = line; return
        mid = (lo + hi) // 2
        if self._f(line, mid) < self._f(cur, mid):
            self.tree[node], line = line, cur
            cur = self.tree[node]
        if lo == hi:
            return
        if self._f(line, lo) < self._f(cur, lo):
            self.add(line, 2 * node, lo, mid)
        elif self._f(line, hi) < self._f(cur, hi):
            self.add(line, 2 * node + 1, mid + 1, hi)

    def query(self, x, node=1, lo=None, hi=None):
        lo = self.lo if lo is None else lo
        hi = self.hi if hi is None else hi
        cur = self.tree.get(node)
        if cur is None:
            return float('inf')
        mid = (lo + hi) // 2
        res = self._f(cur, x)
        if lo == hi:
            return res
        if x <= mid:
            return min(res, self.query(x, 2 * node, lo, mid))
        return min(res, self.query(x, 2 * node + 1, mid + 1, hi))

def min_cost_lichao(nums, cost, k):
    n = len(nums)
    P, C = prefix(nums), prefix(cost)
    lc = LiChao(0, P[n])
    dp = [0] * (n + 1)
    lc.add((-C[0], dp[0] - k * C[0]))
    for r in range(1, n + 1):
        dp[r] = P[r] * C[r] + k * C[n] + lc.query(P[r])
        lc.add((-C[r], dp[r] - k * C[r]))
    return dp[n]

for nums, cost, k, exp in examples:
    assert min_cost_lichao(nums, cost, k) == exp
print('Li Chao khớp ví dụ:', [min_cost_lichao(a, c, k) for a, c, k, _ in examples])

Li Chao khớp ví dụ: [110, 985]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1 | Vét cạn | `O(2^n n)` | Oracle |
| 2 | DP có số đoạn | `O(n^3)` | Hiểu định nghĩa gốc |
| 3 | DP khử thứ tự | `O(n^2)` | **Nộp bài** (`n ≤ 1000`) |
| 4 | CHT deque | `O(n)` | `n` lớn, truy vấn đơn điệu |
| + | Li Chao | `O(n log X)` | Truy vấn không đơn điệu |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minimumCost([1, 1], [1, 1], 1)`.

**Bài 2 (tính tay):** `minimumCost([2, 2, 2], [1, 1, 1], 5)`.

**Bài 3 (điền TODO):** viết `verify_identity(W)` kiểm tra `Σ i·W_i = Σ_j Σ_{i ≥ j} W_i`.

**Bài 4 (tự viết lại):** trả về **cách chia** tối ưu (danh sách đoạn).

**Bài 5 (mini challenge):** biến thể 1 — tối đa `m` đoạn; DP `O(n^2 m)` và kiểm chứng bằng vét cạn.

In [19]:
def verify_identity_todo(W):
    # TODO
    raise NotImplementedError

def best_partition_todo(nums, cost, k):
    # TODO
    raise NotImplementedError

def at_most_m_todo(nums, cost, k, m):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Không chia: `(2 + 1)·2 = 6`. (2) Chia đôi: `(1+1)·1 + (2+2)·1 = 6`. (3) `6`.

**Bài 2.** (1) Không chia: `(6+5)·3 = 33`. (2) Thử các cách khác bằng vét cạn. (3) So sánh: kết quả nhỏ nhất.

**Bài 3.** (1) Vế trái: `sum((i+1)*w)`. (2) Vế phải: `sum(sum(W[j:]))`. (3) Thử ngẫu nhiên.

**Bài 4.** (1) Lưu `arg[r]`. (2) Lần ngược. (3) Tính lại chi phí theo định nghĩa gốc.

**Bài 5.** (1) Thêm chiều số đoạn `j ≤ m`. (2) Dùng định nghĩa gốc với `k·j`. (3) Vét cạn lọc các cách chia có ≤ `m` đoạn.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
a1 = Solution().minimumCost([1, 1], [1, 1], 1)
a2 = Solution().minimumCost([2, 2, 2], [1, 1, 1], 5)
assert a1 == 6 == min_cost_brute([1, 1], [1, 1], 1) and a2 == min_cost_brute([2, 2, 2], [1, 1, 1], 5)

def verify_identity(W):
    return sum((i + 1) * w for i, w in enumerate(W)) == sum(sum(W[j:]) for j in range(len(W)))

assert all(verify_identity([random.randint(0, 50) for _ in range(random.randint(0, 8))]) for _ in range(300))
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3: đẳng thức đúng')

Bài 1: 6 | Bài 2: 33 | Bài 3: đẳng thức đúng


In [21]:
# Bài 4: cách chia tối ưu
def best_partition(nums, cost, k):
    n = len(nums)
    P, C = prefix(nums), prefix(cost)
    dp, arg = [0] + [float('inf')] * n, [0] * (n + 1)
    for r in range(1, n + 1):
        for l in range(r):
            v = dp[l] + P[r] * (C[r] - C[l]) + k * (C[n] - C[l])
            if v < dp[r]:
                dp[r], arg[r] = v, l
    segs, r = [], n
    while r:
        segs.append((arg[r], r - 1)); r = arg[r]
    segs.reverse()
    original = sum((P[r + 1] + k * (i + 1)) * (C[r + 1] - C[l]) for i, (l, r) in enumerate(segs))
    assert original == dp[n]
    return dp[n], [nums[l:r + 1] for l, r in segs]

print('Bài 4:', best_partition(*examples[1][:3]))

Bài 4: (985, [[4, 8, 5, 1], [14, 2, 2], [12, 1]])


In [22]:
# Bài 5: tối đa m đoạn
def at_most_m(nums, cost, k, m):
    n = len(nums)
    P, C = prefix(nums), prefix(cost)
    INF = float('inf')
    f = [[INF] * (n + 1) for _ in range(m + 1)]
    f[0][0] = 0
    for j in range(1, m + 1):
        for r in range(1, n + 1):
            for l in range(r):
                if f[j - 1][l] < INF:
                    f[j][r] = min(f[j][r], f[j - 1][l] + (P[r] + k * j) * (C[r] - C[l]))
    return min(f[j][n] for j in range(1, m + 1))

def at_most_brute(nums, cost, k, m):
    n = len(nums)
    P, C = prefix(nums), prefix(cost)
    best = None
    for mask in range(1 << (n - 1)):
        cuts = [0] + [i + 1 for i in range(n - 1) if mask >> i & 1] + [n]
        if len(cuts) - 1 > m:
            continue
        tot = sum((P[cuts[j + 1]] + k * (j + 1)) * (C[cuts[j + 1]] - C[cuts[j]]) for j in range(len(cuts) - 1))
        best = tot if best is None else min(best, tot)
    return best

random.seed(5)
for _ in range(200):
    n = random.randint(1, 8)
    nums = [random.randint(1, 9) for _ in range(n)]; cost = [random.randint(1, 9) for _ in range(n)]
    k, m = random.randint(1, 9), random.randint(1, n)
    assert at_most_m(nums, cost, k, m) == at_most_brute(nums, cost, k, m)
print('Bài 5: khớp vét cạn; ví dụ 2 với m = 2:', at_most_m(*examples[1][:3], 2))

Bài 5: khớp vét cạn; ví dụ 2 với m = 2: 1029


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
P, C = tổng tiền tố của nums, cost
dp[0] = 0
dp[r] = min_{l<r} dp[l] + P[r]*(C[r]-C[l]) + k*(C[n]-C[l])     # k*i đã đổi thành tổng hậu tố
(CHT: đường y = -C[l]*x + dp[l] - k*C[l], truy vấn x = P[r])
return dp[n]
```

**Checklist:**

- [ ] `nums` lấy **tiền tố** tới cuối đoạn?
- [ ] Phạt `k·i` đổi thành `k·(C[n] − C[l])` cho mỗi đoạn?
- [ ] `dp[0] = 0`, đáp án `dp[n]`?
- [ ] CHT so sánh bằng **nhân chéo số nguyên**?
- [ ] Đối chiếu vét cạn trên mảng nhỏ?